In [ ]:
!pip install -q -U \
    huggingface_hub \
    "pandas==2.2.3" \
    matplotlib

In [ ]:
 
# AIMES PHASE 1
# OPEN-LOOP SINGLE-VALUE ACTIVATION STEERING
#
# Script 4A: Generate baseline + open-loop steered responses
#
 
#
# Intervention:
#
#     h'_{l,t} = h_{l,t} + sign * gamma * u_{k,l}
#
# where:
#
#     u_{k,l} = unit-normalized value direction
#     sign    = +1 : amplify positive pole
#               -1 : move toward negative pole
#     gamma   = fixed open-loop steering strength
#
#
#
# THIS SCRIPT DOES NOT:
#
#   - use J-Lens
#   - use Logit Lens
#   - use Tuned Lens
#   - compute observer scores
#   - use adaptive alpha
#   - implement the AIMES feedback controller
#   - evaluate behavioral value scores
#
#
#
#
# Evaluation input:
#
#   data/evaluation/mfrc_openloop_prompts_v1.csv
#
#
# Value direction input:
#
#   artifacts/value_directions/<model>/v1/directions/
#
#
# Output:
#
#   artifacts/open_loop_steering/<model>/v1/generations/
#
#       open_loop_generations.csv
#       open_loop_generations.jsonl
#       selected_evaluation_prompts.csv
#       open_loop_generation_metadata.json
#
#
# /content is temporary only.
#
 


 
# 0. INSTALL
 

!pip install -q -U \
    transformers \
    accelerate \
    huggingface_hub \
    safetensors \
    "pandas==2.2.3"


 
# 1. IMPORTS
 

import os
import gc
import json
import random
import shutil
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    AutoProcessor,
    Gemma3ForConditionalGeneration,
)

from huggingface_hub import (
    HfApi,
    hf_hub_download,
)

from huggingface_hub.errors import (
    RemoteEntryNotFoundError,
)

from safetensors.torch import (
    load_file,
)

from google.colab import userdata


 
# 2. USER CONFIGURATION
#
# CHANGE EXPERIMENT SETTINGS ONLY IN THIS SECTION.
 


# ------------------------------------------------------------
# 2.1 MODEL
# ------------------------------------------------------------

# Change this line to select the model.

MODEL_KEY = "gemma-4b"
MODEL_KEY = "qwen-4b"
MODEL_KEY = "llama-8b"
MODEL_KEY = "gemma-12b"
MODEL_KEY = "qwen-14b"


# ------------------------------------------------------------
# 2.2 AIMES HUGGING FACE REPOSITORY
# ------------------------------------------------------------

HF_REPO_ID = ""

VERSION = "v1"


# ------------------------------------------------------------
# 2.3 EVALUATION DATASET
# ------------------------------------------------------------

HF_EVAL_DATA_PATH = (
    "data/evaluation/"
    "mfrc_openloop_prompts_v1.csv"
)


# ------------------------------------------------------------
# 2.4 NUMBER OF EVALUATION PROMPTS
# ------------------------------------------------------------

# Number selected PER moral foundation.
#
# None -> use all eligible prompts.

MAX_PROMPTS_PER_FOUNDATION = 50

PROMPT_SAMPLE_SEED = 42


# ------------------------------------------------------------
# 2.5 FOUNDATIONS
# ------------------------------------------------------------

FOUNDATIONS_TO_RUN = [
    "Care",
    "Fairness",
    "Loyalty",
    "Authority",
    "Sanctity",
]


# ------------------------------------------------------------
# 2.6 LAYER SELECTION
# ------------------------------------------------------------

# Options:
#
#   "normalized_grid"
#   "manual"
#   "all"

LAYER_SELECTION_MODE = "normalized_grid"


# Used for normalized_grid.
#
# Approximately equivalent relative depths across models.

NORMALIZED_DEPTHS = [
    0.10,
    0.20,
    0.30,
    0.40,
    0.50,
    0.60,
    0.70,
    0.80,
    0.90,
    1.00,
]


# Used only for:
#
#   LAYER_SELECTION_MODE = "manual"
#
# 1-based transformer layer numbers.

MANUAL_LAYERS = []


# ------------------------------------------------------------
# 2.7 STEERING STRENGTH
# ------------------------------------------------------------

# Main experiment:
#
# use ONE gamma.
#
# Multiple gamma values belong to the later
# steering-strength ablation.

GAMMAS = [
    1.0,
]


# ------------------------------------------------------------
# 2.8 STEERING SIGNS
# ------------------------------------------------------------

# +1:
#   h' = h + gamma*u
#
# -1:
#   h' = h - gamma*u

STEERING_SIGNS = [
    +1,
    -1,
]


# ------------------------------------------------------------
# 2.9 GENERATION SETTINGS
# ------------------------------------------------------------

MAX_NEW_TOKENS = 128


# False -> greedy/deterministic generation.
#
# This is useful for controlled causal comparison.

DO_SAMPLE = False


# Used only if DO_SAMPLE = True.

TEMPERATURE = 0.7

TOP_P = 0.9

TOP_K = 50


REPETITION_PENALTY = 1.0


GENERATION_SEED = 1234


# ------------------------------------------------------------
# 2.10 QWEN3 THINKING
# ------------------------------------------------------------

# Qwen3 non-thinking mode is preferable here because
# we want directly comparable behavioral responses.

QWEN_ENABLE_THINKING = False


# ------------------------------------------------------------
# 2.11 CHECKPOINTING
# ------------------------------------------------------------

# Upload after every N newly generated steering conditions.

SAVE_EVERY_N = 25


# If a partial result exists in HF, continue it.

RESUME_IF_EXISTS = True


# ------------------------------------------------------------
# 2.12 SANITY TEST
# ------------------------------------------------------------

# Before starting the experiment, generate:
#
#   baseline
#   +gamma
#   -gamma
#
# for one prompt.

RUN_STEERING_SANITY_TEST = True


# ------------------------------------------------------------
# 2.13 MODEL/DIRECTION REVISION CHECK
# ------------------------------------------------------------

# Strongly recommended.
#
# Ensures the current model checkpoint is the same
# checkpoint used to construct the directions.

REQUIRE_MODEL_REVISION_MATCH = True


 
# 3. MODEL CONFIGURATION
 

MODEL_CONFIGS = {

    "gemma-4b": {
        "model_id": "google/gemma-3-4b-it",
        "save_name": "gemma-3-4b-it",
        "family": "gemma3",
    },

    "qwen-4b": {
        "model_id": "Qwen/Qwen3-4B",
        "save_name": "qwen3-4b",
        "family": "qwen3",
    },

    "llama-8b": {
        "model_id": "meta-llama/Llama-3.1-8B-Instruct",
        "save_name": "llama-3.1-8b-instruct",
        "family": "llama",
    },

    "gemma-12b": {
        "model_id": "google/gemma-3-12b-it",
        "save_name": "gemma-3-12b-it",
        "family": "gemma3",
    },

    "qwen-14b": {
        "model_id": "Qwen/Qwen3-14B",
        "save_name": "qwen3-14b",
        "family": "qwen3",
    },
}


assert MODEL_KEY in MODEL_CONFIGS, (
    f"Unknown MODEL_KEY: {MODEL_KEY}"
)


MODEL_INFO = MODEL_CONFIGS[
    MODEL_KEY
]


MODEL_ID = MODEL_INFO[
    "model_id"
]


MODEL_SAVE_NAME = MODEL_INFO[
    "save_name"
]


MODEL_FAMILY = MODEL_INFO[
    "family"
]


 
# 4. CANONICAL FOUNDATION ORDER
#
# MUST MATCH SCRIPT 1 DIRECTION GENERATION.
 

FOUNDATIONS = [
    "Care",
    "Fairness",
    "Loyalty",
    "Authority",
    "Sanctity",
]


FOUNDATION_TO_INDEX = {

    foundation: idx

    for idx, foundation
    in enumerate(
        FOUNDATIONS
    )
}


for foundation in FOUNDATIONS_TO_RUN:

    assert foundation in FOUNDATION_TO_INDEX, (
        f"Unknown foundation: {foundation}"
    )


 
# 5. HUGGING FACE AUTHENTICATION
 

HF_TOKEN = userdata.get(
    "HF_TOKEN"
)


assert HF_TOKEN is not None, (
    "HF_TOKEN not found in Colab Secrets."
)


api = HfApi(
    token=HF_TOKEN
)


 
# 6. PRINT CONFIGURATION
 

print("=" * 90)
print("AIMES PHASE 1: OPEN-LOOP STEERING")
print("=" * 90)

print("HF repository      :", HF_REPO_ID)
print("Model              :", MODEL_ID)
print("Model save name    :", MODEL_SAVE_NAME)
print("Family             :", MODEL_FAMILY)
print("Evaluation data    :", HF_EVAL_DATA_PATH)
print("Layer mode         :", LAYER_SELECTION_MODE)
print("Gammas             :", GAMMAS)
print("Steering signs     :", STEERING_SIGNS)
print("Foundations        :", FOUNDATIONS_TO_RUN)
print("Prompts/value      :", MAX_PROMPTS_PER_FOUNDATION)
print("Max new tokens     :", MAX_NEW_TOKENS)
print("Sampling           :", DO_SAMPLE)
print("Revision check     :", REQUIRE_MODEL_REVISION_MATCH)


 
# 7. LOCAL TEMPORARY PATHS
 

LOCAL_ROOT = (
    f"/content/aimes_open_loop/"
    f"{MODEL_SAVE_NAME}/{VERSION}"
)


LOCAL_GENERATION_ROOT = os.path.join(
    LOCAL_ROOT,
    "generations"
)


LOCAL_MODEL_CACHE = os.path.join(
    LOCAL_ROOT,
    "model_cache"
)


# Clean temporary runtime area from a previous execution.

if os.path.exists(
    LOCAL_ROOT
):

    shutil.rmtree(
        LOCAL_ROOT
    )


os.makedirs(
    LOCAL_GENERATION_ROOT,
    exist_ok=True
)


os.makedirs(
    LOCAL_MODEL_CACHE,
    exist_ok=True
)


OUTPUT_CSV = os.path.join(
    LOCAL_GENERATION_ROOT,
    "open_loop_generations.csv"
)


OUTPUT_JSONL = os.path.join(
    LOCAL_GENERATION_ROOT,
    "open_loop_generations.jsonl"
)


SELECTED_PROMPTS_CSV = os.path.join(
    LOCAL_GENERATION_ROOT,
    "selected_evaluation_prompts.csv"
)


METADATA_JSON = os.path.join(
    LOCAL_GENERATION_ROOT,
    "open_loop_generation_metadata.json"
)


 
# 8. HUGGING FACE OUTPUT PATHS
 

HF_OPENLOOP_ROOT = (
    f"artifacts/open_loop_steering/"
    f"{MODEL_SAVE_NAME}/{VERSION}/generations"
)


HF_OUTPUT_CSV = (
    f"{HF_OPENLOOP_ROOT}/"
    "open_loop_generations.csv"
)


HF_OUTPUT_JSONL = (
    f"{HF_OPENLOOP_ROOT}/"
    "open_loop_generations.jsonl"
)


HF_SELECTED_PROMPTS_CSV = (
    f"{HF_OPENLOOP_ROOT}/"
    "selected_evaluation_prompts.csv"
)


HF_METADATA_JSON = (
    f"{HF_OPENLOOP_ROOT}/"
    "open_loop_generation_metadata.json"
)


 
# 9. LOAD MFRC EVALUATION DATA
 

print("\n" + "=" * 90)
print("LOADING EVALUATION DATA")
print("=" * 90)

print("\nHF path:")
print(HF_EVAL_DATA_PATH)


try:

    eval_data_local_path = hf_hub_download(

        repo_id=HF_REPO_ID,

        filename=HF_EVAL_DATA_PATH,

        repo_type="model",

        token=HF_TOKEN,
    )


except RemoteEntryNotFoundError as exc:

    raise RuntimeError(

        "\nMFRC evaluation dataset was not found.\n\n"
        f"Expected:\n"
        f"{HF_REPO_ID}/{HF_EVAL_DATA_PATH}"

    ) from exc


eval_df = pd.read_csv(
    eval_data_local_path
)


REQUIRED_COLUMNS = {
    "prompt_id",
    "foundation",
    "prompt",
}


missing_columns = (

    REQUIRED_COLUMNS
    -
    set(
        eval_df.columns
    )
)


assert not missing_columns, (
    f"Missing evaluation columns: {missing_columns}"
)


print(
    "\nEvaluation dataset shape:",
    eval_df.shape
)


print(
    "Columns:"
)


print(
    list(
        eval_df.columns
    )
)


# Retain requested foundations only.

eval_df = (

    eval_df[
        eval_df[
            "foundation"
        ].isin(
            FOUNDATIONS_TO_RUN
        )
    ]

    .copy()
)


 
# 10. REPRODUCIBLY SELECT PROMPTS
 

if MAX_PROMPTS_PER_FOUNDATION is not None:

    selected_parts = []


    for foundation in FOUNDATIONS_TO_RUN:

        sub = (

            eval_df[
                eval_df[
                    "foundation"
                ]
                ==
                foundation
            ]

            .copy()
        )


        if len(sub) == 0:

            raise RuntimeError(
                f"No evaluation prompts for {foundation}."
            )


        n_select = min(
            MAX_PROMPTS_PER_FOUNDATION,
            len(sub),
        )


        sub = sub.sample(

            n=n_select,

            random_state=(

                PROMPT_SAMPLE_SEED
                +
                FOUNDATION_TO_INDEX[
                    foundation
                ]
            )
        )


        selected_parts.append(
            sub
        )


    eval_df = pd.concat(
        selected_parts,
        ignore_index=True,
    )


else:

    eval_df = (

        eval_df
        .reset_index(
            drop=True
        )
    )


# Stable ordering matters because it determines generation_seed.

eval_df = (

    eval_df

    .sort_values(
        [
            "foundation",
            "prompt_id",
        ]
    )

    .reset_index(
        drop=True
    )
)


# Save exact prompt manifest.

eval_df.to_csv(
    SELECTED_PROMPTS_CSV,
    index=False,
)


print(
    "\nPrompts selected:"
)


print(

    eval_df[
        "foundation"
    ]

    .value_counts()

    .reindex(
        FOUNDATIONS_TO_RUN
    )
)


print(
    "\nTotal selected prompts:",
    len(eval_df)
)


 
# 11. DOWNLOAD VALUE DIRECTIONS
 

direction_filename = (
    f"{MODEL_SAVE_NAME}_"
    f"mft_value_directions_"
    f"{VERSION}.safetensors"
)


HF_DIRECTION_PATH = (
    f"artifacts/value_directions/"
    f"{MODEL_SAVE_NAME}/{VERSION}/directions/"
    f"{direction_filename}"
)


print("\n" + "=" * 90)
print("LOADING VALUE DIRECTIONS")
print("=" * 90)

print(HF_DIRECTION_PATH)


direction_local_path = hf_hub_download(

    repo_id=HF_REPO_ID,

    filename=HF_DIRECTION_PATH,

    repo_type="model",

    token=HF_TOKEN,
)


direction_payload = load_file(
    direction_local_path
)


assert "unit_directions" in direction_payload, (
    "unit_directions missing from direction artifact."
)


UNIT_DIRECTIONS = (

    direction_payload[
        "unit_directions"
    ]

    .float()

    .cpu()
)


print(
    "\nDirection tensor shape:",
    tuple(
        UNIT_DIRECTIONS.shape
    )
)


print(
    "Interpretation:"
)


print(
    "[foundation, layer, hidden_dimension]"
)


assert UNIT_DIRECTIONS.shape[0] == len(FOUNDATIONS)


 
# 12. DOWNLOAD DIRECTION METADATA
#
# Used to verify that directions and steering model
# come from the same model checkpoint.
 

direction_metadata_filename = (
    f"{MODEL_SAVE_NAME}_"
    f"metadata_{VERSION}.json"
)


HF_DIRECTION_METADATA_PATH = (
    f"artifacts/value_directions/"
    f"{MODEL_SAVE_NAME}/{VERSION}/metadata/"
    f"{direction_metadata_filename}"
)


print(
    "\nLoading direction metadata:"
)


print(
    HF_DIRECTION_METADATA_PATH
)


try:

    direction_metadata_local = hf_hub_download(

        repo_id=HF_REPO_ID,

        filename=HF_DIRECTION_METADATA_PATH,

        repo_type="model",

        token=HF_TOKEN,
    )


    with open(
        direction_metadata_local,
        "r",
        encoding="utf-8",
    ) as f:

        DIRECTION_METADATA = json.load(
            f
        )


except RemoteEntryNotFoundError:

    DIRECTION_METADATA = {}


    if REQUIRE_MODEL_REVISION_MATCH:

        raise RuntimeError(

            "Direction metadata is missing, "
            "so model revision compatibility "
            "cannot be verified."
        )


DIRECTION_MODEL_REVISION = (
    DIRECTION_METADATA.get(
        "model_revision"
    )
)


print(
    "Direction revision:",
    DIRECTION_MODEL_REVISION
)


 
# 13. MODEL DTYPE
 

if torch.cuda.is_available():

    if torch.cuda.is_bf16_supported():

        MODEL_DTYPE = torch.bfloat16

    else:

        MODEL_DTYPE = torch.float16


else:

    MODEL_DTYPE = torch.float32


print(
    "\nCompute dtype:",
    MODEL_DTYPE
)


 
# 14. LOAD MODEL / TOKENIZER
#
# Current Transformers uses dtype=.
 

processor = None


if MODEL_FAMILY == "gemma3":

    print(
        "\nLoading Gemma 3 processor..."
    )


    processor = AutoProcessor.from_pretrained(

        MODEL_ID,

        token=HF_TOKEN,

        cache_dir=LOCAL_MODEL_CACHE,
    )


    tokenizer = processor.tokenizer


    print(
        "Loading Gemma 3 model..."
    )


    model = Gemma3ForConditionalGeneration.from_pretrained(

        MODEL_ID,

        token=HF_TOKEN,

        dtype=MODEL_DTYPE,

        device_map="auto",

        cache_dir=LOCAL_MODEL_CACHE,

        low_cpu_mem_usage=True,
    )


else:

    print(
        "\nLoading tokenizer..."
    )


    tokenizer = AutoTokenizer.from_pretrained(

        MODEL_ID,

        token=HF_TOKEN,

        cache_dir=LOCAL_MODEL_CACHE,
    )


    print(
        "Loading causal language model..."
    )


    model = AutoModelForCausalLM.from_pretrained(

        MODEL_ID,

        token=HF_TOKEN,

        dtype=MODEL_DTYPE,

        device_map="auto",

        cache_dir=LOCAL_MODEL_CACHE,

        low_cpu_mem_usage=True,
    )


tokenizer.padding_side = "left"


if tokenizer.pad_token_id is None:

    tokenizer.pad_token = tokenizer.eos_token


model.eval()


 
# 15. VERIFY MODEL ARCHITECTURE
 

config = model.config


if hasattr(
    config,
    "text_config"
):

    text_config = config.text_config

else:

    text_config = config


NUM_LAYERS = int(
    text_config.num_hidden_layers
)


HIDDEN_SIZE = int(
    text_config.hidden_size
)


MODEL_REVISION = getattr(
    config,
    "_commit_hash",
    None,
)


print("\n" + "=" * 90)
print("MODEL ARCHITECTURE")
print("=" * 90)

print("Layers             :", NUM_LAYERS)
print("Hidden size        :", HIDDEN_SIZE)
print("Current revision   :", MODEL_REVISION)
print("Direction revision :", DIRECTION_MODEL_REVISION)


assert (
    UNIT_DIRECTIONS.shape[1]
    ==
    NUM_LAYERS
), (
    "Direction/model layer-count mismatch."
)


assert (
    UNIT_DIRECTIONS.shape[2]
    ==
    HIDDEN_SIZE
), (
    "Direction/model hidden-size mismatch."
)


# ------------------------------------------------------------
# Exact revision check
# ------------------------------------------------------------

if (
    REQUIRE_MODEL_REVISION_MATCH
    and
    DIRECTION_MODEL_REVISION is not None
    and
    MODEL_REVISION is not None
):

    assert (
        MODEL_REVISION
        ==
        DIRECTION_MODEL_REVISION
    ), (

        "\nMODEL REVISION MISMATCH\n\n"
        f"Directions were created with:\n"
        f"  {DIRECTION_MODEL_REVISION}\n\n"
        f"Current model revision:\n"
        f"  {MODEL_REVISION}\n\n"
        "Do not steer using directions from "
        "a different checkpoint."
    )


print(
    "\nModel/direction compatibility: PASS"
)


 
# 16. GET INPUT DEVICE
 

def get_input_device(
    model
):

    try:

        return (
            model
            .get_input_embeddings()
            .weight
            .device
        )

    except Exception:

        pass


    try:

        return (
            model
            .language_model
            .get_input_embeddings()
            .weight
            .device
        )

    except Exception:

        pass


    try:

        return (
            model
            .model
            .language_model
            .get_input_embeddings()
            .weight
            .device
        )

    except Exception:

        pass


    raise RuntimeError(
        "Unable to determine input embedding device."
    )


INPUT_DEVICE = get_input_device(
    model
)


print(
    "Input device:",
    INPUT_DEVICE
)


 
# 17. FIND TRANSFORMER LAYERS
 

def get_transformer_layers(
    model,
    expected_num_layers,
):

    candidates = []


    # Qwen / Llama

    try:

        candidates.append(
            (
                "model.model.layers",
                model.model.layers,
            )
        )

    except Exception:

        pass


    # Gemma-3 multimodal wrapper

    try:

        candidates.append(
            (
                "model.model.language_model.layers",
                model.model.language_model.layers,
            )
        )

    except Exception:

        pass


    try:

        candidates.append(
            (
                "model.language_model.layers",
                model.language_model.layers,
            )
        )

    except Exception:

        pass


    try:

        candidates.append(
            (
                "model.language_model.model.layers",
                model.language_model.model.layers,
            )
        )

    except Exception:

        pass


    try:

        candidates.append(
            (
                "model.model.model.layers",
                model.model.model.layers,
            )
        )

    except Exception:

        pass


    for path, layers in candidates:

        try:

            if len(layers) == expected_num_layers:

                print(
                    "\nTransformer layers found:"
                )


                print(
                    path
                )


                return layers

        except Exception:

            pass


    print(
        "\nCandidate layer containers:"
    )


    for path, layers in candidates:

        try:

            print(
                path,
                len(layers),
            )

        except Exception:

            pass


    raise RuntimeError(
        "Could not identify transformer layer container."
    )


TRANSFORMER_LAYERS = get_transformer_layers(

    model,

    NUM_LAYERS,
)


 
# 18. SELECT INTERVENTION LAYERS
 

def choose_layers(
    num_layers
):

    if LAYER_SELECTION_MODE == "all":

        layers = list(
            range(
                1,
                num_layers + 1,
            )
        )


    elif LAYER_SELECTION_MODE == "manual":

        assert len(MANUAL_LAYERS) > 0, (
            "MANUAL_LAYERS is empty."
        )


        layers = sorted(
            set(
                int(x)
                for x
                in MANUAL_LAYERS
            )
        )


    elif LAYER_SELECTION_MODE == "normalized_grid":

        layers = []


        for depth in NORMALIZED_DEPTHS:

            assert 0 < depth <= 1


            layer = int(
                round(
                    depth
                    *
                    num_layers
                )
            )


            layer = max(
                1,
                min(
                    layer,
                    num_layers,
                )
            )


            layers.append(
                layer
            )


        layers = sorted(
            set(
                layers
            )
        )


    else:

        raise ValueError(
            f"Unknown LAYER_SELECTION_MODE: "
            f"{LAYER_SELECTION_MODE}"
        )


    for layer in layers:

        assert 1 <= layer <= num_layers


    return layers


INTERVENTION_LAYERS = choose_layers(
    NUM_LAYERS
)


print(
    "\nIntervention layers:"
)


print(
    INTERVENTION_LAYERS
)


print(
    "\nRelative depths:"
)


print(
    [
        round(
            layer / NUM_LAYERS,
            3,
        )

        for layer
        in INTERVENTION_LAYERS
    ]
)


 
# 19. ROBUST CHAT TEMPLATE RENDERING
#
# IMPORTANT FIX:
#
# We deliberately perform this in TWO stages:
#
#   messages
#       ↓
#   apply_chat_template(tokenize=False)
#       ↓
#   formatted string
#       ↓
#   tokenizer(..., add_special_tokens=False)
#
# This avoids the BatchEncoding-vs-Tensor issue that caused:
#
#   AttributeError: ndim
#
# in the previous script.
#
# Hugging Face also recommends add_special_tokens=False
# when tokenizing a chat-template string because the template
# already contains the required special tokens.
 

def render_chat_prompt(
    prompt
):

    # --------------------------------------------------------
    # Gemma-3:
    #
    # use processor's multimodal-aware chat template,
    # but provide TEXT ONLY.
    # --------------------------------------------------------

    if MODEL_FAMILY == "gemma3":

        assert processor is not None


        messages = [

            {
                "role": "user",

                "content": [

                    {
                        "type": "text",

                        "text": str(
                            prompt
                        ),
                    }
                ],
            }
        ]


        formatted_prompt = processor.apply_chat_template(

            messages,

            tokenize=False,

            add_generation_prompt=True,
        )


    # --------------------------------------------------------
    # Qwen3 / Llama
    # --------------------------------------------------------

    else:

        messages = [

            {
                "role": "user",

                "content": str(
                    prompt
                ),
            }
        ]


        template_kwargs = {

            "tokenize": False,

            "add_generation_prompt": True,
        }


        if MODEL_FAMILY == "qwen3":

            template_kwargs[
                "enable_thinking"
            ] = QWEN_ENABLE_THINKING


        try:

            formatted_prompt = tokenizer.apply_chat_template(

                messages,

                **template_kwargs,
            )


        except TypeError:

            # Fallback if a Transformers/tokenizer release
            # does not expose enable_thinking.

            template_kwargs.pop(
                "enable_thinking",
                None,
            )


            formatted_prompt = tokenizer.apply_chat_template(

                messages,

                **template_kwargs,
            )


    assert isinstance(
        formatted_prompt,
        str,
    ), (
        "Chat template should return a string when "
        "tokenize=False, but returned "
        f"{type(formatted_prompt)}"
    )


    return formatted_prompt


 
# 20. ENCODE PROMPT
 

def encode_prompt(
    prompt
):

    formatted_prompt = render_chat_prompt(
        prompt
    )


    encoded = tokenizer(

        formatted_prompt,

        return_tensors="pt",

        add_special_tokens=False,
    )


    assert "input_ids" in encoded


    assert torch.is_tensor(
        encoded["input_ids"]
    )


    assert (
        encoded[
            "input_ids"
        ].ndim
        ==
        2
    ), (
        "Expected input_ids with shape "
        "[batch, sequence]."
    )


    if "attention_mask" not in encoded:

        encoded[
            "attention_mask"
        ] = torch.ones_like(
            encoded[
                "input_ids"
            ]
        )


    # Only tensor entries are forwarded to model.generate().

    encoded = {

        key: value.to(
            INPUT_DEVICE
        )

        for key, value
        in encoded.items()

        if torch.is_tensor(
            value
        )
    }


    return encoded


 
# 21. PROMPT ENCODING SANITY CHECK
#
# This specifically verifies the issue that previously failed.
 

encoding_test_prompt = str(
    eval_df.iloc[0][
        "prompt"
    ]
)


encoding_test = encode_prompt(
    encoding_test_prompt
)


print("\n" + "=" * 90)
print("PROMPT ENCODING SANITY CHECK")
print("=" * 90)

print(
    "input_ids type  :",
    type(
        encoding_test[
            "input_ids"
        ]
    )
)


print(
    "input_ids shape :",
    tuple(
        encoding_test[
            "input_ids"
        ].shape
    )
)


print(
    "attention shape :",
    tuple(
        encoding_test[
            "attention_mask"
        ].shape
    )
)


assert (
    encoding_test[
        "input_ids"
    ].shape[0]
    ==
    1
)


del encoding_test


 
# 22. GENERATION KWARGS
 

def get_generation_kwargs():

    kwargs = {

        "max_new_tokens":
            MAX_NEW_TOKENS,

        "do_sample":
            DO_SAMPLE,

        "repetition_penalty":
            REPETITION_PENALTY,

        "pad_token_id":
            tokenizer.pad_token_id,

        "eos_token_id":
            tokenizer.eos_token_id,

        "use_cache":
            True,
    }


    if DO_SAMPLE:

        kwargs.update(
            {
                "temperature":
                    TEMPERATURE,

                "top_p":
                    TOP_P,

                "top_k":
                    TOP_K,
            }
        )


    return kwargs


GENERATION_KWARGS = get_generation_kwargs()


 
# 23. STEERING HOOK
#
# Direction tensor indexing:
#
#   UNIT_DIRECTIONS[k, 0, :]
#
# corresponds to transformer layer 1.
#
# Therefore:
#
#   direction tensor index = layer - 1
#   module index           = layer - 1
#
#
# At each forward pass:
#
#   h[:, -1, :] += sign * gamma * u
#
#
# PREFILL:
#
#   modifies the last prompt-token hidden state.
#
# DECODING WITH KV CACHE:
#
#   the current generated-token hidden state is modified.
#
# Thus the same fixed direction is applied at every
# autoregressive generation step.
 

def make_steering_hook(
    direction,
    gamma,
    sign,
):

    assert sign in (
        -1,
        +1,
    )


    coefficient = (
        float(sign)
        *
        float(gamma)
    )


    def modify_hidden(
        hidden
    ):

        if not torch.is_tensor(
            hidden
        ):

            return hidden


        assert hidden.ndim == 3, (
            f"Expected transformer hidden states "
            f"[B,T,d], got {tuple(hidden.shape)}"
        )


        assert hidden.shape[-1] == HIDDEN_SIZE


        steer = direction.to(

            device=hidden.device,

            dtype=hidden.dtype,
        )


        modified = hidden.clone()


        modified[
            :,
            -1,
            :
        ] = (

            modified[
                :,
                -1,
                :
            ]

            +

            coefficient
            *
            steer
        )


        return modified


    def hook(
        module,
        inputs,
        output,
    ):

        # ----------------------------------------------------
        # Case 1: plain tensor
        # ----------------------------------------------------

        if torch.is_tensor(
            output
        ):

            return modify_hidden(
                output
            )


        # ----------------------------------------------------
        # Case 2: tuple
        #
        # Common decoder-layer output:
        #
        #   (hidden_states, ...)
        # ----------------------------------------------------

        if isinstance(
            output,
            tuple
        ):

            if len(output) == 0:

                return output


            modified_hidden = modify_hidden(
                output[0]
            )


            return (
                modified_hidden,
                *output[1:],
            )


        # ----------------------------------------------------
        # We do NOT silently modify unknown output structures.
        # ----------------------------------------------------

        raise RuntimeError(

            "Unexpected transformer layer output type: "
            f"{type(output)}"
        )


    return hook


 
# 24. RESPONSE GENERATION
 

@torch.inference_mode()
def generate_response(
    prompt,
    seed,
    steering_direction=None,
    intervention_layer=None,
    gamma=None,
    sign=None,
):

    # --------------------------------------------------------
    # Match RNG state for baseline and corresponding
    # intervention whenever sampling is used.
    # --------------------------------------------------------

    random.seed(
        seed
    )


    np.random.seed(
        seed
    )


    torch.manual_seed(
        seed
    )


    if torch.cuda.is_available():

        torch.cuda.manual_seed_all(
            seed
        )


    model_inputs = encode_prompt(
        prompt
    )


    prompt_length = int(
        model_inputs[
            "input_ids"
        ].shape[1]
    )


    hook_handle = None


    # --------------------------------------------------------
    # Register intervention if requested.
    # --------------------------------------------------------

    if steering_direction is not None:

        assert intervention_layer is not None

        assert gamma is not None

        assert sign is not None


        module_index = (
            intervention_layer
            -
            1
        )


        layer_module = TRANSFORMER_LAYERS[
            module_index
        ]


        hook_function = make_steering_hook(

            direction=
                steering_direction,

            gamma=
                gamma,

            sign=
                sign,
        )


        hook_handle = (
            layer_module
            .register_forward_hook(
                hook_function
            )
        )


    try:

        outputs = model.generate(

            **model_inputs,

            **GENERATION_KWARGS,
        )


    finally:

        # Absolutely critical:
        #
        # do not allow a hook from one condition to remain
        # installed for another condition.

        if hook_handle is not None:

            hook_handle.remove()


    assert outputs.ndim == 2


    generated_ids = outputs[
        0,
        prompt_length:
    ]


    response = tokenizer.decode(

        generated_ids,

        skip_special_tokens=True,
    ).strip()


    del model_inputs
    del outputs
    del generated_ids


    gc.collect()


    if torch.cuda.is_available():

        torch.cuda.empty_cache()


    return response


 
# 25. OPEN-LOOP STEERING SANITY TEST
 

if RUN_STEERING_SANITY_TEST:

    print("\n" + "=" * 90)
    print("STEERING SANITY TEST")
    print("=" * 90)


    test_row = eval_df.iloc[0]


    TEST_PROMPT = str(
        test_row[
            "prompt"
        ]
    )


    TEST_FOUNDATION = str(
        test_row[
            "foundation"
        ]
    )


    TEST_LAYER = INTERVENTION_LAYERS[
        len(
            INTERVENTION_LAYERS
        )
        //
        2
    ]


    TEST_GAMMA = GAMMAS[0]


    test_foundation_idx = FOUNDATION_TO_INDEX[
        TEST_FOUNDATION
    ]


    test_direction = UNIT_DIRECTIONS[
        test_foundation_idx,
        TEST_LAYER - 1,
        :
    ]


    test_direction_norm = float(
        torch.linalg.vector_norm(
            test_direction
        )
    )


    assert abs(
        test_direction_norm - 1.0
    ) < 1e-3


    print(
        "\nFoundation:",
        TEST_FOUNDATION
    )


    print(
        "Layer:",
        TEST_LAYER
    )


    print(
        "Gamma:",
        TEST_GAMMA
    )


    print(
        "Direction norm:",
        test_direction_norm
    )


    print(
        "\nGenerating baseline..."
    )


    test_baseline = generate_response(

        prompt=
            TEST_PROMPT,

        seed=
            GENERATION_SEED,
    )


    print(
        "Generating positive steering..."
    )


    test_positive = generate_response(

        prompt=
            TEST_PROMPT,

        seed=
            GENERATION_SEED,

        steering_direction=
            test_direction,

        intervention_layer=
            TEST_LAYER,

        gamma=
            TEST_GAMMA,

        sign=
            +1,
    )


    print(
        "Generating negative steering..."
    )


    test_negative = generate_response(

        prompt=
            TEST_PROMPT,

        seed=
            GENERATION_SEED,

        steering_direction=
            test_direction,

        intervention_layer=
            TEST_LAYER,

        gamma=
            TEST_GAMMA,

        sign=
            -1,
    )


    print("\n" + "-" * 90)

    print("PROMPT:\n")
    print(TEST_PROMPT)

    print("\nBASELINE:\n")
    print(test_baseline)

    print("\nPOSITIVE STEERING:\n")
    print(test_positive)

    print("\nNEGATIVE STEERING:\n")
    print(test_negative)

    print("-" * 90)


 
# 26. DOWNLOAD EXISTING HF CHECKPOINT
 

existing_df = pd.DataFrame()


if RESUME_IF_EXISTS:

    print(
        "\nChecking Hugging Face "
        "for existing generations..."
    )


    try:

        existing_hf_csv = hf_hub_download(

            repo_id=HF_REPO_ID,

            filename=HF_OUTPUT_CSV,

            repo_type="model",

            token=HF_TOKEN,

            force_download=True,
        )


        existing_df = pd.read_csv(
            existing_hf_csv
        )


        # ----------------------------------------------------
        # Protect against accidentally mixing incompatible
        # experiments in one output file.
        # ----------------------------------------------------

        if len(existing_df) > 0:

            if "model_revision" in existing_df.columns:

                old_revisions = set(

                    existing_df[
                        "model_revision"
                    ]

                    .dropna()

                    .astype(str)

                    .unique()
                )


                if (
                    MODEL_REVISION is not None
                    and
                    len(old_revisions) > 0
                ):

                    assert old_revisions == {
                        str(
                            MODEL_REVISION
                        )
                    }, (
                        "Existing HF generation checkpoint "
                        "was created with another model revision."
                    )


        existing_df.to_csv(
            OUTPUT_CSV,
            index=False,
        )


        existing_df.to_json(

            OUTPUT_JSONL,

            orient="records",

            lines=True,

            force_ascii=False,
        )


        print(
            "Existing generation rows:",
            len(existing_df)
        )


    except RemoteEntryNotFoundError:

        print(
            "No existing HF generation checkpoint found."
        )


 
# 27. CONDITION KEY
#
# Unique experimental condition:
#
# prompt x target value x layer x gamma x sign
 

def condition_key(
    prompt_id,
    foundation,
    layer,
    gamma,
    sign,
):

    return (
        str(prompt_id),
        str(foundation),
        int(layer),
        float(gamma),
        int(sign),
    )


completed_keys = set()


if len(existing_df) > 0:

    required_resume_columns = {

        "prompt_id",
        "foundation",
        "layer",
        "gamma",
        "steering_sign",
    }


    assert required_resume_columns.issubset(
        existing_df.columns
    ), (
        "Existing generation checkpoint has "
        "an incompatible schema."
    )


    for _, row in existing_df.iterrows():

        completed_keys.add(

            condition_key(

                row[
                    "prompt_id"
                ],

                row[
                    "foundation"
                ],

                row[
                    "layer"
                ],

                row[
                    "gamma"
                ],

                row[
                    "steering_sign"
                ],
            )
        )


print(
    "\nCompleted steering conditions:",
    len(completed_keys)
)


 
# 28. BASELINE CACHE
#
# Baseline only needs to be generated ONCE for a prompt.
 

baseline_cache = {}


if len(existing_df) > 0:

    if {
        "prompt_id",
        "baseline_response",
    }.issubset(
        existing_df.columns
    ):

        for prompt_id, group in existing_df.groupby(
            "prompt_id"
        ):

            baseline_cache[
                str(
                    prompt_id
                )
            ] = (

                group[
                    "baseline_response"
                ]
                .iloc[0]
            )


 
# 29. LOCAL SAVE FUNCTION
 

def save_results_local(
    new_rows
):

    if len(new_rows) == 0:

        if os.path.exists(
            OUTPUT_CSV
        ):

            return pd.read_csv(
                OUTPUT_CSV
            )

        return pd.DataFrame()


    new_df = pd.DataFrame(
        new_rows
    )


    if os.path.exists(
        OUTPUT_CSV
    ):

        old_df = pd.read_csv(
            OUTPUT_CSV
        )


        combined = pd.concat(

            [
                old_df,
                new_df,
            ],

            ignore_index=True,
        )


    else:

        combined = new_df


    combined = (

        combined

        .drop_duplicates(

            subset=[
                "prompt_id",
                "foundation",
                "layer",
                "gamma",
                "steering_sign",
            ],

            keep="last",
        )

        .reset_index(
            drop=True
        )
    )


    combined.to_csv(
        OUTPUT_CSV,
        index=False,
    )


    combined.to_json(

        OUTPUT_JSONL,

        orient="records",

        lines=True,

        force_ascii=False,
    )


    return combined


 
# 30. HF CHECKPOINT UPLOAD
 

def upload_checkpoint_to_hf():

    if not os.path.exists(
        OUTPUT_CSV
    ):

        return


    print(
        "\nUploading generation checkpoint to HF..."
    )


    api.upload_file(

        path_or_fileobj=
            OUTPUT_CSV,

        path_in_repo=
            HF_OUTPUT_CSV,

        repo_id=
            HF_REPO_ID,

        repo_type=
            "model",

        token=
            HF_TOKEN,
    )


    if os.path.exists(
        OUTPUT_JSONL
    ):

        api.upload_file(

            path_or_fileobj=
                OUTPUT_JSONL,

            path_in_repo=
                HF_OUTPUT_JSONL,

            repo_id=
                HF_REPO_ID,

            repo_type=
                "model",

            token=
                HF_TOKEN,
        )


    print(
        "Checkpoint uploaded."
    )


 
# 31. EXPERIMENT SIZE
 

total_conditions = (

    len(eval_df)

    *
    len(
        INTERVENTION_LAYERS
    )

    *
    len(
        GAMMAS
    )

    *
    len(
        STEERING_SIGNS
    )
)


print("\n" + "=" * 90)
print("OPEN-LOOP EXPERIMENT")
print("=" * 90)

print(
    "Evaluation prompts :",
    len(eval_df)
)


print(
    "Layers             :",
    len(
        INTERVENTION_LAYERS
    )
)


print(
    "Gammas             :",
    len(
        GAMMAS
    )
)


print(
    "Signs              :",
    len(
        STEERING_SIGNS
    )
)


print(
    "Total conditions   :",
    total_conditions
)


print(
    "Already completed  :",
    len(
        completed_keys
    )
)


 
# 32. MAIN GENERATION LOOP
 

new_rows = []

new_since_last_upload = 0

condition_counter = 0


for row_idx, row in eval_df.iterrows():

    prompt_id = row[
        "prompt_id"
    ]


    foundation = str(
        row[
            "foundation"
        ]
    )


    prompt = str(
        row[
            "prompt"
        ]
    )


    foundation_idx = FOUNDATION_TO_INDEX[
        foundation
    ]


    # --------------------------------------------------------
    # Prompt-specific RNG seed.
    #
    # The exact same seed is reused for baseline and
    # every intervention applied to this prompt.
    # --------------------------------------------------------

    prompt_seed = (
        GENERATION_SEED
        +
        int(
            row_idx
        )
    )


    prompt_key = str(
        prompt_id
    )


    # ========================================================
    # BASELINE
    # ========================================================

    if prompt_key in baseline_cache:

        baseline_response = baseline_cache[
            prompt_key
        ]


    else:

        print("\n" + "=" * 80)

        print(
            f"Prompt {row_idx + 1}/"
            f"{len(eval_df)}"
        )


        print(
            "Foundation:",
            foundation
        )


        print(
            "Generating baseline..."
        )


        baseline_response = generate_response(

            prompt=
                prompt,

            seed=
                prompt_seed,
        )


        baseline_cache[
            prompt_key
        ] = baseline_response


    # ========================================================
    # STEERING CONDITIONS
    # ========================================================

    for layer in INTERVENTION_LAYERS:

        direction = UNIT_DIRECTIONS[
            foundation_idx,
            layer - 1,
            :
        ]


        direction_norm = float(
            torch.linalg.vector_norm(
                direction
            )
        )


        assert abs(
            direction_norm
            -
            1.0
        ) < 1e-3, (
            f"Direction is not normalized: "
            f"{direction_norm}"
        )


        relative_depth = (
            layer
            /
            NUM_LAYERS
        )


        for gamma in GAMMAS:

            for sign in STEERING_SIGNS:

                condition_counter += 1


                key = condition_key(

                    prompt_id,

                    foundation,

                    layer,

                    gamma,

                    sign,
                )


                if key in completed_keys:

                    continue


                print(

                    f"\r"
                    f"Condition "
                    f"{condition_counter}/"
                    f"{total_conditions} | "
                    f"{foundation} | "
                    f"L={layer} | "
                    f"d={relative_depth:.3f} | "
                    f"gamma={gamma} | "
                    f"sign={sign:+d}",

                    end="",
                )


                steered_response = generate_response(

                    prompt=
                        prompt,

                    seed=
                        prompt_seed,

                    steering_direction=
                        direction,

                    intervention_layer=
                        layer,

                    gamma=
                        gamma,

                    sign=
                        sign,
                )


                new_rows.append(
                    {

                        # ------------------------------
                        # Model
                        # ------------------------------

                        "model_key":
                            MODEL_KEY,

                        "model_id":
                            MODEL_ID,

                        "model_save_name":
                            MODEL_SAVE_NAME,

                        "model_revision":
                            MODEL_REVISION,

                        "direction_model_revision":
                            DIRECTION_MODEL_REVISION,

                        "version":
                            VERSION,


                        # ------------------------------
                        # Evaluation prompt
                        # ------------------------------

                        "prompt_id":
                            prompt_id,

                        "foundation":
                            foundation,

                        "prompt":
                            prompt,


                        # ------------------------------
                        # Intervention
                        # ------------------------------

                        "layer":
                            int(
                                layer
                            ),

                        "num_layers":
                            NUM_LAYERS,

                        "relative_depth":
                            float(
                                relative_depth
                            ),

                        "gamma":
                            float(
                                gamma
                            ),

                        "steering_sign":
                            int(
                                sign
                            ),

                        "steering_objective":
                            (
                                "amplify"
                                if sign == +1
                                else
                                "suppress"
                            ),

                        "direction_norm":
                            direction_norm,


                        # ------------------------------
                        # Responses
                        # ------------------------------

                        "baseline_response":
                            baseline_response,

                        "steered_response":
                            steered_response,


                        # ------------------------------
                        # Generation settings
                        # ------------------------------

                        "generation_seed":
                            prompt_seed,

                        "max_new_tokens":
                            MAX_NEW_TOKENS,

                        "do_sample":
                            DO_SAMPLE,

                        "temperature":
                            (
                                TEMPERATURE
                                if DO_SAMPLE
                                else None
                            ),

                        "top_p":
                            (
                                TOP_P
                                if DO_SAMPLE
                                else None
                            ),

                        "top_k":
                            (
                                TOP_K
                                if DO_SAMPLE
                                else None
                            ),

                        "repetition_penalty":
                            REPETITION_PENALTY,


                        # ------------------------------
                        # Timestamp
                        # ------------------------------

                        "created_at":
                            datetime.now(
                                timezone.utc
                            ).isoformat(),
                    }
                )


                completed_keys.add(
                    key
                )


                new_since_last_upload += 1


                # --------------------------------------------
                # Periodic checkpoint
                # --------------------------------------------

                if (
                    new_since_last_upload
                    >=
                    SAVE_EVERY_N
                ):

                    save_results_local(
                        new_rows
                    )


                    new_rows = []


                    upload_checkpoint_to_hf()


                    new_since_last_upload = 0


 
# 33. FINAL LOCAL SAVE
 

if len(new_rows) > 0:

    save_results_local(
        new_rows
    )


if not os.path.exists(
    OUTPUT_CSV
):

    raise RuntimeError(
        "No generation output was produced."
    )


final_df = pd.read_csv(
    OUTPUT_CSV
)


print(
    "\n\nGeneration rows:",
    len(final_df)
)


 
# 34. OUTPUT VALIDATION
 

required_output_columns = {

    "model_id",

    "prompt_id",

    "foundation",

    "layer",

    "relative_depth",

    "gamma",

    "steering_sign",

    "baseline_response",

    "steered_response",
}


missing_output_columns = (

    required_output_columns
    -
    set(
        final_df.columns
    )
)


assert not missing_output_columns, (
    f"Missing output columns: "
    f"{missing_output_columns}"
)


# ------------------------------------------------------------
# Check that a given prompt always has one baseline.
# ------------------------------------------------------------

baseline_consistency = (

    final_df

    .groupby(
        "prompt_id"
    )[
        "baseline_response"
    ]

    .nunique()
)


assert (
    baseline_consistency.max()
    <=
    1
), (
    "A prompt has inconsistent baseline responses."
)


print(
    "\nRows by foundation:"
)


print(

    final_df[
        "foundation"
    ]

    .value_counts()

    .reindex(
        FOUNDATIONS_TO_RUN
    )
)


print(
    "\nRows by layer:"
)


print(

    final_df[
        "layer"
    ]

    .value_counts()

    .sort_index()
)


print(
    "\nRows by sign:"
)


print(

    final_df[
        "steering_sign"
    ]

    .value_counts()

    .sort_index()
)


 
# 35. METADATA
 

metadata = {

    "project":
        "AIMES",

    "phase":
        "Phase 1: Open-loop steering",

    "artifact":
        "Open-loop steering generations",

    "version":
        VERSION,

    "created_at":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "hf_repo":
        HF_REPO_ID,

    "model_key":
        MODEL_KEY,

    "model_id":
        MODEL_ID,

    "model_save_name":
        MODEL_SAVE_NAME,

    "model_revision":
        MODEL_REVISION,

    "direction_model_revision":
        DIRECTION_MODEL_REVISION,

    "model_revision_match_required":
        REQUIRE_MODEL_REVISION_MATCH,

    "num_layers":
        NUM_LAYERS,

    "hidden_size":
        HIDDEN_SIZE,

    "foundations":
        FOUNDATIONS_TO_RUN,

    "evaluation_dataset_hf_path":
        HF_EVAL_DATA_PATH,

    "selected_prompt_manifest":
        HF_SELECTED_PROMPTS_CSV,

    "max_prompts_per_foundation":
        MAX_PROMPTS_PER_FOUNDATION,

    "prompt_sample_seed":
        PROMPT_SAMPLE_SEED,

    "layer_selection_mode":
        LAYER_SELECTION_MODE,

    "intervention_layers":
        INTERVENTION_LAYERS,

    "relative_depths":
        [

            layer
            /
            NUM_LAYERS

            for layer
            in INTERVENTION_LAYERS
        ],

    "normalized_depth_targets":
        (
            NORMALIZED_DEPTHS

            if
            LAYER_SELECTION_MODE
            ==
            "normalized_grid"

            else
            None
        ),

    "manual_layers":
        (
            MANUAL_LAYERS

            if
            LAYER_SELECTION_MODE
            ==
            "manual"

            else
            None
        ),

    "gammas":
        GAMMAS,

    "steering_signs":
        STEERING_SIGNS,

    "intervention_definition":
        (
            "h'_{l,t} = h_{l,t} "
            "+ sign * gamma * u_{k,l}"
        ),

    "intervention_location":
        (
            "Final active sequence position "
            "at selected transformer layer "
            "during every model forward pass."
        ),

    "prompt_representation":
        (
            "Instruction-model chat template rendered "
            "to text first, then tokenized with "
            "add_special_tokens=False."
        ),

    "generation": {

        "max_new_tokens":
            MAX_NEW_TOKENS,

        "do_sample":
            DO_SAMPLE,

        "temperature":
            (
                TEMPERATURE
                if DO_SAMPLE
                else None
            ),

        "top_p":
            (
                TOP_P
                if DO_SAMPLE
                else None
            ),

        "top_k":
            (
                TOP_K
                if DO_SAMPLE
                else None
            ),

        "repetition_penalty":
            REPETITION_PENALTY,

        "generation_seed":
            GENERATION_SEED,
    },

    "qwen_enable_thinking":
        (
            QWEN_ENABLE_THINKING

            if
            MODEL_FAMILY
            ==
            "qwen3"

            else
            None
        ),

    "direction_hf_path":
        HF_DIRECTION_PATH,

    "direction_metadata_hf_path":
        HF_DIRECTION_METADATA_PATH,

    "output_hf_root":
        HF_OPENLOOP_ROOT,

    "total_generation_rows":
        int(
            len(final_df)
        ),
}


with open(
    METADATA_JSON,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        metadata,
        f,
        indent=2,
        ensure_ascii=False,
    )


 
# 36. FINAL JSONL
 

final_df.to_json(

    OUTPUT_JSONL,

    orient="records",

    lines=True,

    force_ascii=False,
)


 
# 37. FINAL HF UPLOAD
 

print("\n" + "=" * 90)
print("UPLOADING FINAL PHASE-1 ARTIFACTS")
print("=" * 90)


files_to_upload = [

    (
        OUTPUT_CSV,
        HF_OUTPUT_CSV,
    ),

    (
        OUTPUT_JSONL,
        HF_OUTPUT_JSONL,
    ),

    (
        SELECTED_PROMPTS_CSV,
        HF_SELECTED_PROMPTS_CSV,
    ),

    (
        METADATA_JSON,
        HF_METADATA_JSON,
    ),
]


for local_file, hf_path in files_to_upload:

    print(
        "Uploading:",
        hf_path
    )


    api.upload_file(

        path_or_fileobj=
            local_file,

        path_in_repo=
            hf_path,

        repo_id=
            HF_REPO_ID,

        repo_type=
            "model",

        token=
            HF_TOKEN,
    )


 
# 38. COMPLETENESS CHECK
 

expected_conditions = (

    len(eval_df)

    *
    len(
        INTERVENTION_LAYERS
    )

    *
    len(
        GAMMAS
    )

    *
    len(
        STEERING_SIGNS
    )
)


actual_unique_conditions = (

    final_df[
        [
            "prompt_id",
            "foundation",
            "layer",
            "gamma",
            "steering_sign",
        ]
    ]

    .drop_duplicates()

    .shape[0]
)


print("\n" + "=" * 90)
print("EXPERIMENT COMPLETENESS")
print("=" * 90)

print(
    "Expected conditions :",
    expected_conditions
)


print(
    "Actual conditions   :",
    actual_unique_conditions
)


if (
    actual_unique_conditions
    ==
    expected_conditions
):

    print(
        "\nSTATUS: COMPLETE"
    )


else:

    print(
        "\nSTATUS: PARTIAL"
    )


    print(
        "Rerun with RESUME_IF_EXISTS=True "
        "to continue."
    )


 
# 39. FINAL SUMMARY
 

print("\n" + "=" * 90)
print("AIMES PHASE-1 OPEN-LOOP GENERATION FINISHED")
print("=" * 90)

print("\nModel:")
print(MODEL_ID)

print("\nRevision:")
print(MODEL_REVISION)

print("\nEvaluation prompts:")
print(len(eval_df))

print("\nLayers:")
print(INTERVENTION_LAYERS)

print("\nGammas:")
print(GAMMAS)

print("\nSigns:")
print(STEERING_SIGNS)

print("\nSaved conditions:")
print(len(final_df))

print("\nHF output:")
print(HF_OPENLOOP_ROOT)

print("\nFiles:")
print("  open_loop_generations.csv")
print("  open_loop_generations.jsonl")
print("  selected_evaluation_prompts.csv")
print("  open_loop_generation_metadata.json")

print("\nNext:")
print(
    "Evaluate baseline and steered responses "
    "for target-value change."
)


 
# 40. CLEAN MODEL CACHE
 

del model
del tokenizer
del UNIT_DIRECTIONS


if processor is not None:

    del processor


gc.collect()


if torch.cuda.is_available():

    torch.cuda.empty_cache()


if os.path.exists(
    LOCAL_MODEL_CACHE
):

    shutil.rmtree(
        LOCAL_MODEL_CACHE
    )


print(
    "\nTemporary model cache deleted."
)


print(
    "Done."
)